# Классификация: CC50 выше медианы

Задача: предсказать, превышает ли CC50 медианное значение выборки (424.17 mM).

Сравним модели на полном наборе из 210 признаков и сокращённом наборе из 158 признаков.

## 1. Импорты

In [22]:
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Загрузка и подготовка

In [2]:
raw_data = pd.read_excel("../data/dataset.xlsx")
TARGET_COLUMNS = ["IC50, mM", "CC50, mM", "SI"]

data = raw_data.iloc[:, 1:].copy()
data = data.drop_duplicates()

feature_columns = [col for col in data.columns if col not in TARGET_COLUMNS]
X_full = data[feature_columns]

# Сокращённый набор
constant_mask = X_full.nunique(dropna=False) <= 1

dominant_share = X_full.apply(
    lambda column: column.value_counts(
        dropna=False,
        normalize=True,
    ).iloc[0]
)

near_constant_mask = dominant_share >= 0.95

X_reduced = X_full.loc[
    :,
    ~(constant_mask | near_constant_mask),
]

threshold = data["CC50, mM"].median()
y = (data["CC50, mM"] > threshold).astype(int)

print(f"Полный набор: {X_full.shape[1]} признаков")
print(f"Сокращённый: {X_reduced.shape[1]} признаков")
print(f"Класс 0: {(y == 0).sum()}, класс 1: {(y == 1).sum()} ({y.mean():.1%} положительных)")

Полный набор: 210 признаков
Сокращённый: 158 признаков
Класс 0: 485, класс 1: 484 (49.9% положительных)


## 3. Разбиение

Используем стратифицированное групповое разбиение. Объекты с одинаковыми значениями молекулярных дескрипторов относятся к одной группе и не могут одновременно попасть в обучение и тест.

Для подбора гиперпараметров также используем групповую стратифицированную кросс-валидацию.

In [3]:
groups = pd.util.hash_pandas_object(X_full, index=False)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X_full, y, groups=groups)
)

Xf_train = X_full.iloc[train_idx]
Xf_test = X_full.iloc[test_idx]

Xr_train = X_reduced.iloc[train_idx]
Xr_test = X_reduced.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print(f"Обучение: {len(Xf_train)}, тест: {len(Xf_test)}")

Обучение: 774, тест: 195


## 4. Метрики

Три метрики, каждая про своё:

- Accuracy — доля правильных ответов. При равных классах случайное угадывание даст 0.5
- F1 — среднее гармоническое точности и полноты положительного класса. Метрика учитывает как пропущенные положительные объекты, так и ложные положительные предсказания
- ROC-AUC — площадь под ROC-кривой. Показывает, насколько хорошо модель разделяет классы независимо от порога. 0.5 — случайно, 1.0 — идеально

Подбираем гиперпараметры по F1. Для сравнения используем baseline, который относит все объекты к положительному классу

## 5. Логистическая регрессия

Линейный классификатор. Признаки масштабируем, пропуски заполняем медианой.

In [4]:
lr_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000))
lr_params = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}

lr_search = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search.fit(Xf_train, y_train, groups=groups_train)

lr_cv = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(lr_cv)
print(f"Лучший C: {lr_search.best_params_['logisticregression__C']}, CV F1: {lr_search.best_score_:.4f}")

Полный набор


,C,CV F1
0,0.01,0.731155
1,0.10,0.738303
2,1.00,0.715202
3,10.00,0.700786
4,100.00,0.695352


Лучший C: 0.1, CV F1: 0.7383


In [5]:
lr_pred = lr_search.predict(Xf_test)
lr_proba = lr_search.predict_proba(Xf_test)[:, 1]
lr_full = {"name": "LR (полный)", "acc": accuracy_score(y_test, lr_pred), "f1": f1_score(y_test, lr_pred), "roc": roc_auc_score(y_test, lr_proba)}
print(f"Test Accuracy: {lr_full['acc']:.3f}, Test F1: {lr_full['f1']:.3f}, ROC-AUC: {lr_full['roc']:.3f}")

Test Accuracy: 0.738, Test F1: 0.754, ROC-AUC: 0.828


In [6]:
lr_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000)), lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search2.fit(Xr_train, y_train, groups=groups_train)
lr_cv2 = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(lr_cv2)
print(f"Лучший C: {lr_search2.best_params_['logisticregression__C']}, CV F1: {lr_search2.best_score_:.4f}")

Сокращённый набор


,C,CV F1
0,0.01,0.710322
1,0.10,0.719353
2,1.00,0.720370
3,10.00,0.723560
4,100.00,0.700642


Лучший C: 10, CV F1: 0.7236


In [7]:
lr_pred2 = lr_search2.predict(Xr_test)
lr_proba2 = lr_search2.predict_proba(Xr_test)[:, 1]
lr_reduced = {"name": "LR (сокращённый)", "acc": accuracy_score(y_test, lr_pred2), "f1": f1_score(y_test, lr_pred2), "roc": roc_auc_score(y_test, lr_proba2)}
print(f"Test Accuracy: {lr_reduced['acc']:.3f}, Test F1: {lr_reduced['f1']:.3f}, ROC-AUC: {lr_reduced['roc']:.3f}")

Test Accuracy: 0.733, Test F1: 0.748, ROC-AUC: 0.818


На полном наборе логистическая регрессия получила F1 = 0.754, на сокращённом — 0.748. Результаты близкие, но полный набор немного лучше по всем трём метрикам.

## 6. K ближайших соседей

Голосование K соседей. StandardScaler критичен — без него признаки с большим размахом задавят остальные. Подбираем число соседей и тип взвешивания на обоих наборах.

In [8]:
knn_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier())
knn_params = {"kneighborsclassifier__n_neighbors": [3, 5, 7, 10, 15, 20], "kneighborsclassifier__weights": ["uniform", "distance"]}

knn_search = GridSearchCV(knn_pipe, knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search.fit(Xf_train, y_train, groups=groups_train)

knn_cv = pd.DataFrame({"n_neighbors": knn_search.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(knn_cv.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search.best_params_}, CV F1: {knn_search.best_score_:.4f}")

Полный набор


,n_neighbors,weights,CV F1
9,15,distance,0.746348
8,15,uniform,0.743532
7,10,distance,0.741915
4,7,uniform,0.738891
0,3,uniform,0.738067
3,5,distance,0.731929
2,5,uniform,0.730255
11,20,distance,0.730141
1,3,distance,0.729645
5,7,distance,0.725254


Лучшие: {'kneighborsclassifier__n_neighbors': 15, 'kneighborsclassifier__weights': 'distance'}, CV F1: 0.7463


In [9]:
knn_pred = knn_search.predict(Xf_test)
knn_proba = knn_search.predict_proba(Xf_test)[:, 1]
knn_full = {"name": "KNN (полный)", "acc": accuracy_score(y_test, knn_pred), "f1": f1_score(y_test, knn_pred), "roc": roc_auc_score(y_test, knn_proba)}
print(f"Test Accuracy: {knn_full['acc']:.3f}, Test F1: {knn_full['f1']:.3f}, ROC-AUC: {knn_full['roc']:.3f}")

Test Accuracy: 0.738, Test F1: 0.751, ROC-AUC: 0.806


In [10]:
knn_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier()), knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search2.fit(Xr_train, y_train, groups=groups_train)
knn_cv2 = pd.DataFrame({"n_neighbors": knn_search2.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search2.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(knn_cv2.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search2.best_params_}, CV F1: {knn_search2.best_score_:.4f}")

Сокращённый набор


,n_neighbors,weights,CV F1
3,5,distance,0.753510
0,3,uniform,0.751252
1,3,distance,0.750090
2,5,uniform,0.749159
4,7,uniform,0.748189
5,7,distance,0.745617
7,10,distance,0.745474
9,15,distance,0.744542
8,15,uniform,0.743554
11,20,distance,0.742038


Лучшие: {'kneighborsclassifier__n_neighbors': 5, 'kneighborsclassifier__weights': 'distance'}, CV F1: 0.7535


In [11]:
knn_pred2 = knn_search2.predict(Xr_test)
knn_proba2 = knn_search2.predict_proba(Xr_test)[:, 1]
knn_reduced = {"name": "KNN (сокращённый)", "acc": accuracy_score(y_test, knn_pred2), "f1": f1_score(y_test, knn_pred2), "roc": roc_auc_score(y_test, knn_proba2)}
print(f"Test Accuracy: {knn_reduced['acc']:.3f}, Test F1: {knn_reduced['f1']:.3f}, ROC-AUC: {knn_reduced['roc']:.3f}")

Test Accuracy: 0.718, Test F1: 0.729, ROC-AUC: 0.806


KNN получил F1 = 0.751 на полном наборе и 0.729 на сокращённом. Удаление признаков ухудшило F1 и Accuracy, при этом ROC-AUC практически не изменился.

## 7. Случайный лес

Ансамбль деревьев. Не требует масштабирования. Расширенная сетка: 48 комбинаций, полный перебор на обоих наборах.

In [12]:
rf_pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
rf_params = {"randomforestclassifier__n_estimators": [100, 200, 300, 500], "randomforestclassifier__max_depth": [10, 20, 30, None], "randomforestclassifier__min_samples_leaf": [1, 3, 5], "randomforestclassifier__max_features": ["sqrt", "log2", None]}

rf_search = GridSearchCV(rf_pipe, rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search.fit(Xf_train, y_train, groups=groups_train)

rf_cv = pd.DataFrame({"n_estimators": rf_search.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(rf_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search.best_params_}, CV F1: {rf_search.best_score_:.4f}")

Полный набор


,n_estimators,max_depth,min_samples_leaf,max_features,CV F1
15,500,10,1,log2,0.753859
120,100,None,1,log2,0.751634
84,100,30,1,log2,0.751634
74,300,30,1,sqrt,0.751597
110,300,None,1,sqrt,0.751597
38,300,20,1,sqrt,0.751552
39,500,20,1,sqrt,0.750155
48,100,20,1,log2,0.749811
16,100,10,3,log2,0.749653
14,300,10,1,log2,0.749519


Лучшие: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__max_features': 'log2', 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 500}, CV F1: 0.7539


In [13]:
rf_pred = rf_search.predict(Xf_test)
rf_proba = rf_search.predict_proba(Xf_test)[:, 1]
rf_full = {"name": "RF (полный)", "acc": accuracy_score(y_test, rf_pred), "f1": f1_score(y_test, rf_pred), "roc": roc_auc_score(y_test, rf_proba)}
print(f"Test Accuracy: {rf_full['acc']:.3f}, Test F1: {rf_full['f1']:.3f}, ROC-AUC: {rf_full['roc']:.3f}")

Test Accuracy: 0.769, Test F1: 0.774, ROC-AUC: 0.830


In [14]:
rf_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42)), rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search2.fit(Xr_train, y_train, groups=groups_train)
rf_cv2 = pd.DataFrame({"n_estimators": rf_search2.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search2.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search2.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search2.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(rf_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search2.best_params_}, CV F1: {rf_search2.best_score_:.4f}")

Сокращённый набор


,n_estimators,max_depth,min_samples_leaf,max_features,CV F1
16,100,10,3,log2,0.758387
39,500,20,1,sqrt,0.748587
17,200,10,3,log2,0.748045
75,500,30,1,sqrt,0.747470
111,500,None,1,sqrt,0.747470
74,300,30,1,sqrt,0.747432
110,300,None,1,sqrt,0.747432
0,100,10,1,sqrt,0.746290
3,500,10,1,sqrt,0.746128
50,300,20,1,log2,0.746123


Лучшие: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__max_features': 'log2', 'randomforestclassifier__min_samples_leaf': 3, 'randomforestclassifier__n_estimators': 100}, CV F1: 0.7584


In [15]:
rf_pred2 = rf_search2.predict(Xr_test)
rf_proba2 = rf_search2.predict_proba(Xr_test)[:, 1]
rf_reduced = {"name": "RF (сокращённый)", "acc": accuracy_score(y_test, rf_pred2), "f1": f1_score(y_test, rf_pred2), "roc": roc_auc_score(y_test, rf_proba2)}
print(f"Test Accuracy: {rf_reduced['acc']:.3f}, Test F1: {rf_reduced['f1']:.3f}, ROC-AUC: {rf_reduced['roc']:.3f}")

Test Accuracy: 0.749, Test F1: 0.751, ROC-AUC: 0.825


Случайный лес получил F1 = 0.774 на полном наборе и 0.751 на сокращённом. Полный набор показал лучший результат по всем трём метрикам.

## 8. Градиентный бустинг

Последовательное построение деревьев, каждое исправляет ошибки предыдущих. RandomizedSearchCV по 20 комбинациям на обоих наборах.

In [16]:
gb_pipe = make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42))
gb_params = {"gradientboostingclassifier__n_estimators": [100, 200, 300], "gradientboostingclassifier__max_depth": [3, 5, 7], "gradientboostingclassifier__learning_rate": [0.01, 0.05, 0.1]}

gb_search = RandomizedSearchCV(gb_pipe, gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search.fit(Xf_train, y_train, groups=groups_train)

gb_cv = pd.DataFrame({"n_estimators": gb_search.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(gb_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search.best_params_}, CV F1: {gb_search.best_score_:.4f}")

Полный набор


,n_estimators,max_depth,learning_rate,CV F1
18,300,5,0.10,0.739233
19,100,3,0.10,0.735332
2,100,3,0.05,0.734609
17,200,7,0.10,0.733500
15,200,5,0.10,0.730517
1,200,5,0.05,0.728341
5,300,3,0.05,0.727741
7,300,7,0.05,0.727714
13,300,3,0.01,0.727451
10,200,3,0.01,0.726600


Лучшие: {'gradientboostingclassifier__n_estimators': 300, 'gradientboostingclassifier__max_depth': 5, 'gradientboostingclassifier__learning_rate': 0.1}, CV F1: 0.7392


In [17]:
gb_pred = gb_search.predict(Xf_test)
gb_proba = gb_search.predict_proba(Xf_test)[:, 1]
gb_full = {"name": "GBT (полный)", "acc": accuracy_score(y_test, gb_pred), "f1": f1_score(y_test, gb_pred), "roc": roc_auc_score(y_test, gb_proba)}
print(f"Test Accuracy: {gb_full['acc']:.3f}, Test F1: {gb_full['f1']:.3f}, ROC-AUC: {gb_full['roc']:.3f}")

Test Accuracy: 0.744, Test F1: 0.745, ROC-AUC: 0.807


In [18]:
gb_search2 = RandomizedSearchCV(make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42)), gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search2.fit(Xr_train, y_train, groups=groups_train)
gb_cv2 = pd.DataFrame({"n_estimators": gb_search2.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search2.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search2.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(gb_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search2.best_params_}, CV F1: {gb_search2.best_score_:.4f}")

Сокращённый набор


,n_estimators,max_depth,learning_rate,CV F1
1,200,5,0.05,0.731033
15,200,5,0.10,0.730600
5,300,3,0.05,0.729198
18,300,5,0.10,0.728778
10,200,3,0.01,0.728524
3,100,5,0.10,0.728246
14,100,7,0.05,0.724467
7,300,7,0.05,0.722819
8,100,5,0.05,0.722572
12,300,5,0.01,0.722138


Лучшие: {'gradientboostingclassifier__n_estimators': 200, 'gradientboostingclassifier__max_depth': 5, 'gradientboostingclassifier__learning_rate': 0.05}, CV F1: 0.7310


In [19]:
gb_pred2 = gb_search2.predict(Xr_test)
gb_proba2 = gb_search2.predict_proba(Xr_test)[:, 1]
gb_reduced = {"name": "GBT (сокращённый)", "acc": accuracy_score(y_test, gb_pred2), "f1": f1_score(y_test, gb_pred2), "roc": roc_auc_score(y_test, gb_proba2)}
print(f"Test Accuracy: {gb_reduced['acc']:.3f}, Test F1: {gb_reduced['f1']:.3f}, ROC-AUC: {gb_reduced['roc']:.3f}")

Test Accuracy: 0.764, Test F1: 0.758, ROC-AUC: 0.827


Градиентный бустинг получил F1 = 0.745 на полном наборе и 0.758 на сокращённом. Это единственная модель, качество которой улучшилось после удаления признаков.

## 9. Сравнение

Сравним модели на полном и сокращённом наборах с наивным baseline.

In [23]:
baseline_pred = np.ones(len(y_test), dtype=int)
baseline_proba = np.ones(len(y_test))

baseline = {
    "name": "Baseline (все 1)",
    "acc": accuracy_score(y_test, baseline_pred),
    "f1": f1_score(y_test, baseline_pred),
    "roc": roc_auc_score(y_test, baseline_proba),
}

print(
    f"Baseline | Accuracy: {baseline['acc']:.3f}, "
    f"F1: {baseline['f1']:.3f}, "
    f"ROC-AUC: {baseline['roc']:.3f}"
)

Baseline | Accuracy: 0.503, F1: 0.669, ROC-AUC: 0.500


In [24]:
res_full = pd.DataFrame(
    [baseline, lr_full, knn_full, rf_full, gb_full]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_full.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("210 признаков")
display(res_full.round(3))

res_reduced = pd.DataFrame(
    [baseline, lr_reduced, knn_reduced, rf_reduced, gb_reduced]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_reduced.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("158 признаков")
display(res_reduced.round(3))

=== 210 признаков ===


,Модель,Accuracy,F1,ROC-AUC
0,RF (полный),0.769,0.774,0.830
1,LR (полный),0.738,0.754,0.828
2,KNN (полный),0.738,0.751,0.806
3,GBT (полный),0.744,0.745,0.807
4,Baseline (все 1),0.503,0.669,0.500


=== 158 признаков ===


,Модель,Accuracy,F1,ROC-AUC
0,GBT (сокращённый),0.764,0.758,0.827
1,RF (сокращённый),0.749,0.751,0.825
2,LR (сокращённый),0.733,0.748,0.818
3,KNN (сокращённый),0.718,0.729,0.806
4,Baseline (все 1),0.503,0.669,0.500


## 10. Выводы

Лучший результат на тестовой выборке показал случайный лес на полном наборе признаков: F1 = 0.774, Accuracy = 0.769 и ROC-AUC = 0.830.

На сокращённом наборе лучший результат получил градиентный бустинг: F1 = 0.758, Accuracy = 0.764 и ROC-AUC = 0.827.

Baseline, который относит все объекты к положительному классу, получил F1 = 0.669. Все обученные модели превзошли его, поэтому молекулярные дескрипторы содержат полезную информацию для классификации CC50.

Удаление константных и почти константных признаков не дало общего улучшения. Качество градиентного бустинга выросло, но результаты логистической регрессии, KNN и случайного леса ухудшились.

Для итогового решения выбираем случайный лес на полном наборе признаков. Групповое разбиение исключает попадание объектов с одинаковыми дескрипторами одновременно в обучение и тест.